# bird-watcher: GPU validation
Включите Internet и GPU T4 x2. Этот ноутбук запускает эталон, упражнения находятся
в 01_resnet и 02_vlm. Запуск скачивает официальный CUB и веса Qwen.

In [ ]:
import os, subprocess, sys
from pathlib import Path
assert Path("/kaggle/working").is_dir(), "This setup cell is for Kaggle only"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
repo = Path("/kaggle/working/bird-watcher")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/Mixanik-43/bird-watcher.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[vlm]", "pytest>=8"], check=True)
# Kaggle currently bundles torchao 0.10; PEFT 0.20 rejects it even for plain LoRA.
# No quantization is used here. Only remove it in this disposable Kaggle environment.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU"
print(torch.__version__, torch.cuda.get_device_name(), torch.cuda.get_device_properties(0).total_memory / 1e9)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
if not Path("data/cub8/manifest.jsonl").exists():
    import shutil
    attached = list(Path("/kaggle/input").rglob("manifest.jsonl"))
    if len(attached) == 1:
        shutil.copytree(attached[0].parent, "data/cub8", dirs_exist_ok=True)
    else:
        subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/train_resnet.py", "--epochs", "8"], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/train_vlm.py", "--steps", "50", "--eval-size", "24"], check=True)